# Day 26: ViT Attention Visualization

> **Goal:** Visualize what the Vision Transformer's attention heads learn — which patches attend to which, and how attention patterns differ from CNN feature maps.

---

## Why Visualize ViT Attention?

Unlike CNNs where we look at **feature maps** and **Grad-CAM**, for ViTs we can directly inspect the **attention weights** — they tell us exactly which patches the model considers when making predictions.

### What to expect:
- **Early layers:** Local attention (nearby patches attend to each other)
- **Deep layers:** Global attention (semantically related patches connect)
- **Different heads:** Each head captures different relationships

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

### Rebuilding the ViT Architecture

Below we rebuild the exact same Vision Transformer from Day 25. The key addition here is a `return_weights` option in the attention layer — this lets us **extract the raw attention matrices** during inference, which is what we'll visualize.

The ViT has a `get_attention_maps()` method that runs a forward pass while collecting attention weights from every layer. Each attention map has shape `(batch, heads, seq_len, seq_len)`, where `seq_len = num_patches + 1` (the +1 is the `[CLS]` token).

In [ ]:
# Rebuild ViT (same architecture as Day 25)
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=128):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size
        self.patch_proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model) * 0.02)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        return self.dropout(x + self.pos_embed)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, return_weights=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)
        out = (attn @ V).transpose(1, 2).reshape(B, N, C)
        out = self.proj_drop(self.proj(out))
        if return_weights:
            return out, attn
        return out


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, expansion=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * expansion), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model), nn.Dropout(dropout))

    def forward(self, x, return_weights=False):
        if return_weights:
            attn_out, weights = self.attn(self.norm1(x), return_weights=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return (x, weights) if return_weights else x


class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 d_model=128, num_heads=4, num_layers=6, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, 4, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, num_classes))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

    def get_attention_maps(self, x):
        """Extract attention weights from all layers."""
        attn_maps = []
        x = self.patch_embed(x)
        for block in self.blocks:
            x, weights = block(x, return_weights=True)
            attn_maps.append(weights)
        return attn_maps  # List of (B, heads, N, N)

---
## 1. Train a Quick ViT (or Load Checkpoint)

In [ ]:
# Data
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25)])
test_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_data = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
classes = train_data.classes

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)

# Quick train
model = ViT(img_size=32, patch_size=4, d_model=128, num_heads=4, num_layers=6, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

model.train()
for epoch in range(15):
    correct, total = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = loss_fn(out, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}: Train Acc = {correct/total:.2%}")

print("Training complete!")

---
## 2. Visualize Attention Maps Per Layer

In [ ]:
def unnormalize(img, mean=CIFAR_MEAN, std=CIFAR_STD):
    img = img.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1)


# Get a test image
raw_data = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
img, label = raw_data[7]
print(f"Image class: {classes[label]}")

# Get attention maps
model.eval()
with torch.no_grad():
    attn_maps = model.get_attention_maps(img.unsqueeze(0).to(device))

print(f"\nNumber of layers: {len(attn_maps)}")
print(f"Attention map shape: {attn_maps[0].shape}  (batch, heads, seq, seq)")

### Understanding CLS Token Attention

The `[CLS]` token is the special "summary" token at position 0. It doesn't correspond to any image patch — instead, it **attends to all patches** and aggregates information for classification.

By looking at `attention[0, head, 0, 1:]` we can see **which patches the CLS token pays attention to** at each layer and head. We reshape the resulting 64 values back into an 8×8 grid (matching the 8×8 patch layout) and overlay it on the original image as a heatmap.

- **Warm colors** (red/yellow) = high attention = the model focuses here
- **Cool colors** (blue) = low attention = less important for the prediction

In [ ]:
# Visualize attention from [CLS] token to all patches at each layer
# CLS token is position 0, so attn[0, head, 0, 1:] gives attention to each patch
patch_grid = 8  # 32/4 = 8×8 grid

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for layer_idx in range(6):
    row, col = divmod(layer_idx, 6)
    attn = attn_maps[layer_idx][0]  # (heads, N+1, N+1)

    # Average across heads, get CLS attention to patches
    cls_attn = attn.mean(dim=0)[0, 1:].cpu()  # (64,)
    cls_attn = cls_attn.reshape(patch_grid, patch_grid).numpy()

    ax = axes.flat[layer_idx]
    ax.imshow(unnormalize(img).permute(1, 2, 0))
    ax.imshow(cls_attn, alpha=0.6, cmap="jet",
              extent=[0, 32, 32, 0])
    ax.set_title(f"Layer {layer_idx + 1}", fontsize=10)
    ax.axis("off")

# Also show original
for j in range(6, 12):
    axes.flat[j].axis("off")

axes[1, 0].imshow(unnormalize(img).permute(1, 2, 0))
axes[1, 0].set_title("Original", fontsize=10)

plt.suptitle("[CLS] Token Attention Across Layers (Avg Heads)", fontsize=14)
plt.tight_layout()
plt.show()

### Per-Head Attention Patterns

Multi-head attention splits the representation into multiple "heads," each learning different attention patterns. Below we visualize each head's CLS attention **independently** at the last layer.

You may notice that different heads focus on different parts of the image — one head might attend to edges, another to texture regions, and another to the object center. This **head specialization** is a key advantage of multi-head attention over single-head attention.

In [ ]:
# Attention per head at a specific layer
layer_idx = 5  # Last layer
attn = attn_maps[layer_idx][0].cpu()  # (heads, N+1, N+1)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

# Original
axes[0].imshow(unnormalize(img).permute(1, 2, 0))
axes[0].set_title("Original")
axes[0].axis("off")

for h in range(4):
    cls_attn = attn[h, 0, 1:].reshape(patch_grid, patch_grid).numpy()
    axes[h+1].imshow(unnormalize(img).permute(1, 2, 0))
    axes[h+1].imshow(cls_attn, alpha=0.6, cmap="jet", extent=[0, 32, 32, 0])
    axes[h+1].set_title(f"Head {h+1}")
    axes[h+1].axis("off")

plt.suptitle(f"Layer {layer_idx+1}: [CLS] Attention Per Head", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Attention Rollout

Simple per-layer attention can be misleading. **Attention rollout** combines attention across layers by multiplying attention matrices, accounting for residual connections.

In [ ]:
def attention_rollout(attn_maps):
    """
    Compute attention rollout across layers.
    attn_maps: list of (1, heads, N, N) tensors
    """
    # Average across heads
    result = torch.eye(attn_maps[0].shape[-1])

    for attn in attn_maps:
        attn_avg = attn[0].mean(dim=0).cpu()  # (N, N)
        # Add residual connection (identity)
        attn_avg = 0.5 * attn_avg + 0.5 * torch.eye(attn_avg.shape[0])
        # Normalize
        attn_avg = attn_avg / attn_avg.sum(dim=-1, keepdim=True)
        result = attn_avg @ result

    return result


rollout = attention_rollout(attn_maps)
cls_rollout = rollout[0, 1:].reshape(patch_grid, patch_grid).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(unnormalize(img).permute(1, 2, 0))
axes[0].set_title("Original")

axes[1].imshow(cls_rollout, cmap="hot")
axes[1].set_title("Attention Rollout")

axes[2].imshow(unnormalize(img).permute(1, 2, 0))
axes[2].imshow(cls_rollout, alpha=0.6, cmap="jet", extent=[0, 32, 32, 0])
axes[2].set_title("Overlay")

for ax in axes: ax.axis("off")
plt.suptitle("[CLS] Attention Rollout", fontsize=14)
plt.tight_layout()
plt.show()

### Comparing Across Multiple Images

A single image can be misleading. Below we apply attention rollout to several test images from different classes to see if the model consistently focuses on the relevant object. If the model is well-trained, the attention maps should highlight the **main subject** of each image regardless of its class.

---
## 4. Compare Multiple Images

In [ ]:
# Attention rollout for several test images
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
row_labels = ["Original", "Attention\nRollout", "Overlay"]

sample_indices = [7, 100, 250, 500, 800, 1200]

model.eval()
for col, idx in enumerate(sample_indices):
    img_i, label_i = raw_data[idx]
    img_show = unnormalize(img_i).permute(1, 2, 0)

    with torch.no_grad():
        maps = model.get_attention_maps(img_i.unsqueeze(0).to(device))

    rollout = attention_rollout(maps)
    cls_attn = rollout[0, 1:].reshape(patch_grid, patch_grid).numpy()

    axes[0, col].imshow(img_show)
    axes[0, col].set_title(classes[label_i], fontsize=10)

    axes[1, col].imshow(cls_attn, cmap="hot")

    axes[2, col].imshow(img_show)
    axes[2, col].imshow(cls_attn, alpha=0.6, cmap="jet", extent=[0, 32, 32, 0])

for ax in axes.flat:
    ax.axis("off")
for row, lbl in enumerate(row_labels):
    axes[row, 0].set_ylabel(lbl, fontsize=11, rotation=0, labelpad=60, va="center")

plt.suptitle("ViT Attention Rollout on CIFAR-10 Test Images", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Positional Embedding Analysis (After Training)

In [ ]:
# Visualize learned positional embeddings after training
pos_embed = model.patch_embed.pos_embed.detach().cpu().squeeze()  # (N+1, d_model)

# Cosine similarity between positional embeddings
patch_pos = pos_embed[1:]  # Skip CLS
cos_sim = F.cosine_similarity(patch_pos.unsqueeze(0), patch_pos.unsqueeze(1), dim=-1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Show similarity of specific patches with all others
for i, patch_idx in enumerate([0, 9, 27, 63]):
    row_pos = patch_idx // patch_grid
    col_pos = patch_idx % patch_grid
    sim = cos_sim[patch_idx].reshape(patch_grid, patch_grid).numpy()

    axes[0, i].imshow(sim, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[0, i].set_title(f"Patch ({row_pos},{col_pos})", fontsize=10)
    axes[0, i].axis("off")

# Full similarity matrix
axes[1, 0].imshow(cos_sim.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1, 0].set_title("Full Similarity Matrix")

# Distance-based similarity
distances = []
similarities = []
for i in range(64):
    for j in range(i+1, 64):
        ri, ci = i // 8, i % 8
        rj, cj = j // 8, j % 8
        dist = math.sqrt((ri-rj)**2 + (ci-cj)**2)
        distances.append(dist)
        similarities.append(cos_sim[i, j].item())

axes[1, 1].scatter(distances, similarities, alpha=0.3, s=5)
axes[1, 1].set_xlabel("Spatial Distance")
axes[1, 1].set_ylabel("Cosine Similarity")
axes[1, 1].set_title("Position vs Similarity")
axes[1, 1].grid(True, alpha=0.3)

for j in range(2, 4):
    axes[1, j].axis("off")

plt.suptitle("Learned Positional Embeddings", fontsize=14)
plt.tight_layout()
plt.show()

### What the Plots Below Show

- **Top row (per-patch similarity):** For four different patch positions, we show the cosine similarity of that patch's positional embedding with all other patches. Nearby patches should have high similarity (red), confirming that the model has **learned spatial structure** purely from data.
- **Bottom-left (full matrix):** The full 64×64 similarity matrix — look for block structure indicating spatial awareness.
- **Bottom-right (distance vs. similarity):** A scatter plot showing that closer patches tend to have more similar positional embeddings — evidence that the model has rediscovered 2D spatial relationships despite being given no explicit spatial information!

---
## 🧪 Exercises

### Exercise 1: Attention to specific patches
Pick a non-CLS patch (e.g., patch 10). Visualize which other patches it attends to at each layer.

### Exercise 2: Head specialization
For each head in each layer, compute the average "locality" — do some heads attend more locally than others?

### Exercise 3: Attention diversity
Compute the entropy of each head's attention distribution. Higher entropy = more uniform attention. How does it change across layers?

### Exercise 4: Grad-CAM on ViT
Implement gradient-based attention by computing gradients of the predicted class w.r.t. the attention weights.

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **[CLS] attention** | Shows which patches the model uses for classification |
| **Attention rollout** | Combine attention across layers for a global view |
| **Head specialization** | Different heads learn different attention patterns |
| **Positional embeddings** | Learn spatial relationships (nearby patches = similar) |
| **Early vs late layers** | Early = local patterns, Late = global semantics |

**Tomorrow:** CNN vs ViT — head-to-head comparison on the same dataset!